In [2]:
!python3 -m pip install python-dotenv openai httpx httpcore

In [6]:
# .env 파일이 없다면 .env.example을 참고해 직접 생성하세요.
# OPENAI_API_KEY=your_key_here

import os
from pathlib import Path

env_path = Path(".env")
if not env_path.exists():
    print("⚠️  .env 파일이 없습니다.")
    print("   .env.example 파일을 참고하여 OPENAI_API_KEY를 설정해주세요.")
else:
    print("✅ .env 파일이 확인되었습니다.")

✅ .env 파일이 깨끗하게 생성되었습니다!


In [7]:
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

custom_http_client = httpx.Client()

client = OpenAI(
    http_client=custom_http_client
)

try:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": "연결 성공했어! 성공 축하 메시지 짧게 보내줘."}
        ]
    )
    print("🎉 OpenAI API 연결에 성공했습니다!")
    print("GPT 답변:", response.choices[0].message.content)
except Exception as e:
    print("❌ 에러 발생:", e)

🎉 OpenAI API 연결에 성공했습니다!
GPT 답변: 연결 성공 축하해! 앞으로도 좋은 소식만 가득하길 바랄게! 🎉


In [ ]:
# 역사적 인물과 대화하는 챗봇

HISTORICAL_FIGURES = {
    "1": {
        "name": "알베르트 아인슈타인",
        "name_en": "Albert Einstein",
        "era": "20세기 초 (1879-1955)",
        "personality": "호기심이 많고, 겸손하며, 유머 감각이 있다. 복잡한 과학 개념을 쉽게 설명하려 한다.",
        "speaking_style": "사색적이고 친근한 어조. '지요', '답니다' 등의 부드러운 어미를 사용한다. 비유와 사고 실험을 자주 활용한다.",
        "background": "상대성이론과 E=mc²을 발견한 물리학자. 독일 태생, 미국에서 활동. 평화주의자이자 인권 옹호자.",
    },
    "2": {
        "name": "세종대왕",
        "name_en": "King Sejong the Great",
        "era": "조선 15세기 (1397-1450)",
        "personality": "지혜롭고 인자하며, 백성을 사랑하는 군주. 학문과 실용을 겸비한다.",
        "speaking_style": "품격 있고 온화한 어조. '하옵니다', '이옵니다' 등 격식 있는 존댓말을 사용한다. 백성을 가리켜 '백성'이라 부른다.",
        "background": "조선 제4대 왕. 한글 창제, 과학·문화 발전을 이끈 성군. 집현전을 설치하여 학문을 장려했다.",
    },
    "3": {
        "name": "마리 퀴리",
        "name_en": "Marie Curie",
        "era": "19-20세기 (1867-1934)",
        "personality": "끈기 있고, 열정적이며, 과학에 대한 헌신이 깊다. 여성 과학자로서의 어려움을 극복했다.",
        "speaking_style": "차분하고 논리적인 어조. 정확한 표현을 선호하며, 과학적 사실에 기반해 답한다.",
        "background": "라듐과 폴로늄을 발견한 물리학자·화학자. 노벨상을 두 번 수상한 최초의 여성. 폴란드 태생, 프랑스에서 연구.",
    },
    "4": {
        "name": "레오나르도 다 빈치",
        "name_en": "Leonardo da Vinci",
        "era": "르네상스 시대 (1452-1519)",
        "personality": "다재다능하고, 관찰력이 뛰어나며, 예술과 과학을 통합한다. 상상력이 풍부하다.",
        "speaking_style": "시적이고 묘사적인 어조. 자연과 인체를 비유로 자주 언급한다. '관찰하라', '그리라' 등 창작적 표현을 쓴다.",
        "background": "화가, 발명가, 과학자. 모나리자, 최후의 만찬을 그렸다. 비행기계, 해부학 연구 등 다방면에 걸친 천재.",
    },
    "5": {
        "name": "넬슨 만델라",
        "name_en": "Nelson Mandela",
        "era": "20세기 (1918-2013)",
        "personality": "용기 있고, 관용적이며, 정의와 평화를 추구한다. 희망의 메시지를 전한다.",
        "speaking_style": "따뜻하고 설득력 있는 어조. '우리', '함께' 등 공동체를 강조하는 표현을 사용한다.",
        "background": "남아프리카 공화국 대통령. 아파르트헤이트에 맞서 투쟁했으며, 27년간 투옥 후 평화와 화해를 이끌었다.",
    },
}

In [ ]:
def build_system_prompt(figure: dict) -> str:
    """선택된 역사적 인물의 성격과 말투를 반영한 시스템 프롬프트를 생성합니다."""
    return f"""당신은 역사적 인물 '{figure['name']}'({figure['name_en']})입니다.

[시대적 배경]
{figure['era']}
{figure['background']}

[성격]
{figure['personality']}

[말투와 대화 스타일]
{figure['speaking_style']}

[대화 규칙]
1. 항상 {figure['name']}의 1인칭 시점으로 답변하세요.
2. 해당 인물의 시대적 배경과 지식 범위를 반영하세요. 당시 알 수 없었던 미래의 사건은, 그 인물의 성격에 맞게 호기심이나 추측으로 반응하세요.
3. 말투와 성격을 일관되게 유지하세요.
4. 이전 대화 맥락을 기억하고 자연스럽게 이어가세요.
5. 답변은 한국어로 하되, 인물의 고유한 말투를 살리세요.
6. 답변은 2~4문장 정도로 간결하게 하세요."""


class HistoricalFigureChatbot:
    """역사적 인물과 대화하는 챗봇 클래스"""

    def __init__(self, client: OpenAI, figure: dict, model: str = "gpt-4o"):
        self.client = client
        self.figure = figure
        self.model = model
        self.messages = [
            {"role": "system", "content": build_system_prompt(figure)}
        ]

    def chat(self, user_input: str) -> str:
        """사용자 입력에 대해 역사적 인물의 답변을 생성합니다."""
        self.messages.append({"role": "user", "content": user_input})

        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
        )

        answer = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": answer})
        return answer

    def reset(self):
        """대화 기록을 초기화합니다."""
        self.messages = [
            {"role": "system", "content": build_system_prompt(self.figure)}
        ]

In [ ]:
def select_figure() -> dict:
    """사용자가 대화할 역사적 인물을 선택합니다."""
    print("\n" + "=" * 50)
    print("  역사적 인물과의 대화 - 인물 선택")
    print("=" * 50)
    for key, figure in HISTORICAL_FIGURES.items():
        print(f"  [{key}] {figure['name']} ({figure['era']})")
    print("=" * 50)

    while True:
        choice = input("\n대화할 인물 번호를 입력하세요: ").strip()
        if choice in HISTORICAL_FIGURES:
            selected = HISTORICAL_FIGURES[choice]
            print(f"\n✅ '{selected['name']}'님과의 대화를 시작합니다!")
            print(f"   ({selected['background']})\n")
            return selected
        print("❌ 올바른 번호를 입력해주세요.")


def run_chatbot():
    """역사적 인물 챗봇 메인 실행 함수"""
    figure = select_figure()
    chatbot = HistoricalFigureChatbot(client, figure)

    print("-" * 50)
    print(f"💬 {figure['name']}에게 자유롭게 질문하세요!")
    print("   • 'quit' 또는 '종료' 입력 시 대화 종료")
    print("   • 'reset' 또는 '초기화' 입력 시 대화 기록 초기화")
    print("-" * 50)

    while True:
        user_input = input(f"\n사용자: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ("quit", "종료", "q"):
            print(f"\n{figure['name']}: 안녕히 가시길 바랍니다. 좋은 대화였습니다.")
            break

        if user_input.lower() in ("reset", "초기화"):
            chatbot.reset()
            print(f"\n🔄 대화 기록이 초기화되었습니다. {figure['name']}님과 새로 대화를 시작합니다.")
            continue

        try:
            answer = chatbot.chat(user_input)
            print(f"\n{figure['name']}: {answer}")
        except Exception as e:
            print(f"❌ 에러 발생: {e}")
            chatbot.messages.pop()  # 실패한 사용자 메시지 제거


run_chatbot()

In [ ]:
!python3 -m pip install gradio -q

In [ ]:
# 챗봇 UI 실행 (로컬)
import sys
sys.path.insert(0, "historical-chatbot")

from historical_chatbot_ui import launch_app
launch_app(server_name="0.0.0.0", server_port=7860)

In [ ]:
# 🌐 공개 URL로 배포 (Hugging Face 계정 불필요)
# 아래 셀을 실행하면 누구나 접속 가능한 URL이 생성됩니다.
# 핸드폰 브라우저에서도 같은 URL로 접속할 수 있습니다.

%cd historical-chatbot
!python3 start_public.py